# Hybrid DeBERTa + Qwen 14B - All-in-One
- Trains DeBERTa with multiple seeds
- Identifies hard examples (top 10% by std across seeds)
- Runs Qwen 14B inference only on hard examples
- Blends: 0.5 * deberta_prob + 0.5 * qwen_prob for hard examples

## Part 1: DeBERTa Training

In [1]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

Using Python 3.11.13 environment at: /usr
Resolved 168 packages in 593ms                                       
   Building deepspeed==0.17.4                                          
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                   

In [2]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import multiprocessing as mp
from transformers import get_linear_schedule_with_warmup
%env TOKENIZERS_PARALLELISM= 'false'

%env KAGGLE_IS_COMPETITION_RERUN= 'true'

env: TOKENIZERS_PARALLELISM='false'
env: KAGGLE_IS_COMPETITION_RERUN='true'


In [3]:
MAX_LEN = 256
BATCH_SIZE = 24
EPOCHS =  1#5
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
SEEDS = [42, 123]
HARD_EXAMPLE_RATIO = 0.10

In [4]:
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [5]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [6]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [7]:
test_df = pd.read_csv(test_path)

augmented_train = add_data(df)
augmented_test = add_data(test_df)

augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

Before:(10185, 2)
After: (1875, 2)


,text,label,rule,body,rule_id
0,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...","\n\nIf you have some free time on your hands, ...",0
1,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\n\nplease visit http://www.shifadental.net/te...,0
2,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n\nSD | [ English Stream 1 Arsenal vs Tottenh...,0
3,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n**HD** ENG [ 1080P HD Amazing] :- [USTREAM E...,0
4,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\nFree http://forums.airdroid.com/viewtopic.ph...,0


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [9]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    for seed in SEEDS:
        train_data, val_data = train_test_split(
            augmented_df,
            test_size=0.2,
            stratify=augmented_df["rule"],
            random_state=seed
        )
        train_data.to_csv(f'fixed_train_split_seed_{seed}.csv', index=False)
        val_data.to_csv(f'fixed_val_split_seed_{seed}.csv', index=False)
        print(f'Seed {seed} splits saved: train={len(train_data)}, val={len(val_data)}')

Seed 42 splits saved: train=1500, val=375
Seed 123 splits saved: train=1500, val=375


In [10]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [11]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [12]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Training'):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [13]:
def validate(model, loader, device):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            rule_ids = batch["rule_ids"]

            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)

    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)

    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}

    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]

        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan

    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0

    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [14]:
def train_model_seed(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed}] Training on {device}")

    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')

    train_ds = JigsawDataset(
        train_data['text'].tolist(),
        train_data['label'].tolist(),
        train_data['rule_id'].tolist(),
        tokenizer, MAX_LEN
    )

    val_ds = JigsawDataset(
        val_data['text'].tolist(),
        val_data['label'].tolist(),
        val_data['rule_id'].tolist(),
        tokenizer, MAX_LEN
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = JigsawModel(MODEL_PATH).to(device)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_auc = 0
    best_loss= None
    for epoch in range(EPOCHS):
        print(f"[Seed {seed}] Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)

        print(f"[Seed {seed}] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss= val_loss
            torch.save(model.state_dict(), f"model_seed_{seed}.bin")

    print(f"[Seed {seed}] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_seed_{seed}.json', 'w') as f:
        json.dump({'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return seed, best_auc

In [15]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import torch.multiprocessing as mp
    mp.set_start_method('fork', force=True)

    processes = []
    for idx, seed in enumerate(SEEDS):
       gpu_id = idx % torch.cuda.device_count()
       p = mp.Process(target=train_model_seed, args=(seed, gpu_id))
       p.start()
       processes.append(p)

    for p in processes:
       p.join()

    import json
    results = []
    for seed in SEEDS:
      with open(f'results_seed_{seed}.json', 'r') as f:
          results.append(json.load(f))

    aucs = [r['best_auc'] for r in results]
    losses = [r['best_loss'] for r in results]

    print(f"AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"Loss: {np.mean(losses):.4f} ± {np.std(losses):.4f}")

    print("All models trained!")

[Seed 42] Training on cuda:0
[Seed 123] Training on cuda:1


2025-10-20 15:14:42.398422: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-20 15:14:42.398422: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760973282.626758     227 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760973282.626743     226 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760973282.693826     227 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1760973282.693844     226 cuda_blas.cc:1

[Seed 42] Epoch 1/1


Training:   0%|          | 0/63 [00:00<?, ?it/s]

[Seed 123] Epoch 1/1


Training: 100%|██████████| 63/63 [01:14<00:00,  1.18s/it]


[Seed 42] Loss: 0.6789, Val Loss: 0.6767, Val AUC: 0.6893
[Seed 123] Loss: 0.6782, Val Loss: 0.6505, Val AUC: 0.7554
[Seed 42] Best validation AUC: 0.6893
[Seed 123] Best validation AUC: 0.7554
AUC: 0.7224 ± 0.0331
Loss: 0.6636 ± 0.0131
All models trained!


## Part 2: DeBERTa Inference & Identify Hard Examples

In [16]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]

    test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    all_preds = []

    for seed in SEEDS:
        device = torch.device("cuda:0")
        model = JigsawModel(MODEL_PATH).to(device)
        model.load_state_dict(torch.load(f"model_seed_{seed}.bin", map_location=device))
        model.eval()

        test_preds = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Inference seed {seed}"):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(ids, mask)
                test_preds.extend(torch.sigmoid(logits).cpu().numpy())

        all_preds.append(test_preds)
        
        seed_df = pd.DataFrame({
            'row_id': df_test['row_id'],
            f'pred_seed_{seed}': test_preds
        })
        seed_df.to_csv(f'test_preds_seed_{seed}.csv', index=False)
        print(f"Saved predictions for seed {seed}")

    seed_preds = np.array(all_preds)
    deberta_mean = seed_preds.mean(axis=0)
    deberta_std = seed_preds.std(axis=0)

    print(f"\nMean predictions - min: {deberta_mean.min():.4f}, max: {deberta_mean.max():.4f}")
    print(f"Std - min: {deberta_std.min():.4f}, max: {deberta_std.max():.4f}, mean: {deberta_std.mean():.4f}")

2025-10-20 15:16:26.638821: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760973386.660355      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760973386.667390      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Inference seed 42: 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]


Saved predictions for seed 42


Inference seed 123: 100%|██████████| 1/1 [00:00<00:00,  3.04it/s]

Saved predictions for seed 123

Mean predictions - min: 0.3880, max: 0.5413
Std - min: 0.0035, max: 0.0864, mean: 0.0334


In [17]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    n_hard = int(len(deberta_mean) * HARD_EXAMPLE_RATIO)
    hard_indices = np.argsort(deberta_std)[-n_hard:]

    print(f"Total examples: {len(deberta_mean)}")
    print(f"Hard examples (top {HARD_EXAMPLE_RATIO*100}% by std): {n_hard}")
    print(f"Std threshold: {deberta_std[hard_indices].min():.4f}")

    df_test['deberta_mean'] = deberta_mean
    df_test['deberta_std'] = deberta_std
    df_test['is_hard'] = False
    df_test.loc[hard_indices, 'is_hard'] = True

    print(f"\nHard examples: {df_test['is_hard'].sum()}")

    df_hard = df_test[df_test['is_hard']].copy()
    print(f"Hard examples for 14B inference: {len(df_hard)}")
    
    df_hard.to_csv('hard_examples.csv', index=False)
    print("Saved hard examples to hard_examples.csv")

Total examples: 10
Hard examples (top 10.0% by std): 1
Std threshold: 0.0864

Hard examples: 1
Hard examples for 14B inference: 1
Saved hard examples to hard_examples.csv


## Part 3: Write 14B Inference Script & Run as Subprocess

In [18]:
%%writefile qwen_14b_inference.py
#!/usr/bin/env python3
"""
Standalone Qwen 14B inference script
Reads hard examples CSV, runs inference, saves results
"""

import os
import sys
import pandas as pd
import numpy as np
import torch
import vllm
from vllm.lora.request import LoRARequest
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from scipy.special import softmax

def main():
    MODEL_NAME = "/kaggle/input/qwen2.5/transformers/14b-instruct-gptq-int4/1"
    LORA_PATH = "/kaggle/input/lora_14b_gptq_1epoch_r32/keras/default/1"

    print("=" * 80)
    print("Starting Qwen 14B Inference")
    print("=" * 80)

    print("\n[1/5] Loading hard examples...")
    df_hard = pd.read_csv('hard_examples.csv')
    print(f"Loaded {len(df_hard)} hard examples")
    print(f"Columns: {df_hard.columns.tolist()}")

    print("\n[2/5] Initializing Qwen 14B model...")
    llm = vllm.LLM(
        MODEL_NAME,
        quantization='gptq',
        tensor_parallel_size=torch.cuda.device_count(),
        gpu_memory_utilization=0.95,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=4096,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=32
    )

    tokenizer = llm.get_tokenizer()
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
    print("Model loaded successfully")

    print("\n[3/5] Creating prompts...")
    SYS_PROMPT = """
You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
"""

    prompts = []
    for i, row in df_hard.iterrows():
        text = f"""
r/{row['subreddit']}
Rule: {row['rule']}

1) {row['positive_example_1']}
Violation: Yes

2) {row['positive_example_2']}
Violation: Yes

3) {row['negative_example_1']}
Violation: No

4) {row['negative_example_2']}
Violation: No

5) {row['body']}
"""

        messages = [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": text}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Answer:"
        prompts.append(prompt)

    print(f"Created {len(prompts)} prompts")

    print("\n[4/5] Running inference...")
    outputs = llm.generate(
        prompts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )
    print(f"Generated {len(outputs)} outputs")

    print("\n[5/5] Processing results...")
    logprobs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]

    logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
    qwen_probs = logit_matrix.apply(lambda x: softmax(x.values), axis=1, result_type="expand")
    qwen_probs.columns = ['Yes', 'No']
    qwen_probs['qwen_prob'] = qwen_probs['Yes']

    df_hard['qwen_prob'] = qwen_probs['qwen_prob'].values

    df_hard.to_csv('hard_examples_with_qwen.csv', index=False)
    print(f"\nSaved results to hard_examples_with_qwen.csv")
    print(f"Qwen prob stats:")
    print(qwen_probs['qwen_prob'].describe())

    print("\n" + "=" * 80)
    print("Qwen 14B Inference Complete!")
    print("=" * 80)

if __name__ == "__main__":
    main()

Writing qwen_14b_inference.py


In [19]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    print("GPU memory cleared before 14B inference")

GPU memory cleared before 14B inference


## Part 4: Run 14B Inference as Subprocess

In [20]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("Running Qwen 14B inference as subprocess...")
    !VLLM_USE_V1=0 python qwen_14b_inference.py
    print("\n14B inference completed!")

Running Qwen 14B inference as subprocess...
Starting Qwen 14B Inference

[1/5] Loading hard examples...
Loaded 1 hard examples
Columns: ['row_id', 'body', 'rule', 'subreddit', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2', 'text', 'deberta_mean', 'deberta_std', 'is_hard']

[2/5] Initializing Qwen 14B model...
2025-10-20 15:16:41.557387: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760973401.585574     544 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760973401.592612     544 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 10-20 15:16:46 [__init__.py:235] Automatically detected platform cuda.
`torch_dt

In [21]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_hard_with_qwen = pd.read_csv('hard_examples_with_qwen.csv')
    print(f"Loaded {len(df_hard_with_qwen)} hard examples with Qwen predictions")
    print(f"\nQwen prob stats:")
    print(df_hard_with_qwen['qwen_prob'].describe())

Loaded 1 hard examples with Qwen predictions

Qwen prob stats:
count    1.000000
mean     0.060975
std           NaN
min      0.060975
25%      0.060975
50%      0.060975
75%      0.060975
max      0.060975
Name: qwen_prob, dtype: float64


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


## Part 5: Blend & Submit

In [22]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_hard_with_qwen['blended_prob'] = 0.5 * df_hard_with_qwen['deberta_mean'] + 0.5 * df_hard_with_qwen['qwen_prob']

    print(f"Blended {len(df_hard_with_qwen)} hard examples")
    print(f"\nComparison (first 10):")
    print(df_hard_with_qwen[['row_id', 'deberta_mean', 'deberta_std', 'qwen_prob', 'blended_prob']].head(10))

Blended 1 hard examples

Comparison (first 10):
   row_id  deberta_mean  deberta_std  qwen_prob  blended_prob
0    2036      0.392389     0.086435   0.060975      0.226682


In [23]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_test['final_pred'] = df_test['deberta_mean']

    for idx, row in df_hard_with_qwen.iterrows():
        df_test.loc[idx, 'final_pred'] = row['blended_prob']

    print(f"Final predictions created")
    print(f"Using blended predictions for {df_test['is_hard'].sum()} hard examples")
    print(f"Using DeBERTa predictions for {(~df_test['is_hard']).sum()} easy examples")

Final predictions created
Using blended predictions for 1 hard examples
Using DeBERTa predictions for 9 easy examples


/tmp/ipykernel_36/1772759429.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.2266819697925832' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df_test.loc[idx, 'final_pred'] = row['blended_prob']


In [24]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_test['final_pred'] = df_test['deberta_mean']

    for idx, row in df_hard.iterrows():
        df_test.loc[idx, 'final_pred'] = row['blended_prob']

    print(f"Final predictions created")
    print(f"Using blended predictions for {df_test['is_hard'].sum()} hard examples")
    print(f"Using DeBERTa predictions for {(~df_test['is_hard']).sum()} easy examples")

KeyError: 'blended_prob'

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = df_test[['row_id', 'final_pred']].copy()
    submission.columns = ['row_id', 'rule_violation']
    submission.to_csv('submission.csv', index=False)

    print(f"Submission shape: {submission.shape}")
    print(f"\nSubmission stats:")
    print(submission['rule_violation'].describe())
else:
    !touch submission.csv

In [ ]:
!head submission.csv

## Part 6: Analysis

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("Hard Examples - Prediction Changes:")
    df_hard_with_qwen['diff'] = df_hard_with_qwen['blended_prob'] - df_hard_with_qwen['deberta_mean']
    print(f"\nMean change: {df_hard_with_qwen['diff'].mean():.4f}")
    print(f"Std of change: {df_hard_with_qwen['diff'].std():.4f}")
    print(f"Max increase: {df_hard_with_qwen['diff'].max():.4f}")
    print(f"Max decrease: {df_hard_with_qwen['diff'].min():.4f}")

    print("\nTop 5 examples where Qwen disagreed most with DeBERTa:")
    print(df_hard_with_qwen.nlargest(5, 'diff')[['row_id', 'deberta_mean', 'qwen_prob', 'blended_prob', 'diff']])